<a href="https://colab.research.google.com/github/caffein1371/Carrier-Owl/blob/master/atma1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
import polars as pl

In [3]:
!apt-get -y install fonts-ipafont

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-ipafont is already the newest version (00303-21ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.


In [4]:
!apt-get -y install fonts-ipafont-gothic

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-ipafont-gothic is already the newest version (00303-21ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.


In [5]:
!fc-list | grep "IPA"

/usr/share/fonts/opentype/ipafont-mincho/ipam.ttf: IPAMincho,IPA明朝:style=Regular
/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf: IPAPGothic,IPA Pゴシック:style=Regular
/usr/share/fonts/opentype/ipafont-mincho/ipamp.ttf: IPAPMincho,IPA P明朝:style=Regular
/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf: IPAGothic,IPAゴシック:style=Regular
/usr/share/fonts/truetype/fonts-japanese-mincho.ttf: IPAMincho,IPA明朝:style=Regular
/usr/share/fonts/truetype/fonts-japanese-gothic.ttf: IPAGothic,IPAゴシック:style=Regular


In [6]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# フォントのパスを指定
font_path = "/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf"  # ここを適宜変更
font_prop = fm.FontProperties(fname=font_path)

# Matplotlib のデフォルトフォントを設定
plt.rcParams["font.family"] = font_prop.get_name()

print(f"使用フォント: {font_prop.get_name()}")  # どのフォントが設定されたか確認


使用フォント: IPAGothic


# EDA

In [7]:
#オンライン EC サイト上の購買ログデータです。オフライン購買データ (train_session.csv / train_log.csv 他) とユーザー ID や購入セッション ID で紐づかないことに注意してください。
ec_df = pd.read_csv('/content/drive/MyDrive/atma/ec_log.csv')
#商品を一意に特定することができるコード (jan code) に対して、その商品の属性を記載しているデータです。
jan_df = pd.read_csv('/content/drive/MyDrive/atma/jan.csv')
#提出データの情報
test_session_df = pd.read_csv('/content/drive/MyDrive/atma/test_session.csv')
#学習期間の完全なオフライン購買データ。どの購入セッションで何を何円で何個かったか、の情報が記入されています
train_log_df = pd.read_csv('/content/drive/MyDrive/atma/train_log.csv')
#学習・テスト期間の購入セッションのメタ情報。学習期間とテスト期間は日付で分割されています。 2024-10-31 までが学習期間、それ以降がテスト期間です。
train_session_df = pd.read_csv('/content/drive/MyDrive/atma/train_session.csv')

train_target_df = pd.read_csv('/content/drive/MyDrive/atma/train_target.csv')

<ipython-input-7-b106b868f3a2>:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  ec_df = pd.read_csv('/content/drive/MyDrive/atma/ec_log.csv')


In [8]:
target_col_list = ['チョコレート', 'ビール', 'ヘアケア', '米（5㎏以下）']

In [9]:
def remove_negative_purchase(train_session_df, train_log_df, jan_df):
    """
    下記を考慮して、返品されているsessionはややこしいのですべて除外する
    https://www.guruguru.science/competitions/26/discussions/1bc20930-cf84-43ed-93d0-c6ea64e504ee/
    """
    # 売上数量が0以下のデータを抽出（返品）
    neg_train_log_df = train_log_df[train_log_df["売上数量"] <= 0].copy()

    # 売上数量が正のデータのみ残す
    train_log_df = train_log_df[train_log_df["売上数量"] > 0].copy()

    # 返品された session を取得
    ignore_sessions = neg_train_log_df.merge(jan_df, on="JAN", how="inner")
    ignore_sessions = ignore_sessions[ignore_sessions["カテゴリ名"].isin(target_col_list)][["session_id"]]

    # `how="anti"` に相当する処理（除外する session_id を持つ行を削除）
    train_session_df = train_session_df.merge(ignore_sessions, on="session_id", how="left", indicator=True)
    train_session_df = train_session_df[train_session_df["_merge"] == "left_only"].drop(columns=["_merge"])

    return train_log_df, train_session_df

# 適用
train_log_df, train_session_df = remove_negative_purchase(
    train_session_df,
    train_log_df,
    jan_df,
)

In [10]:
from IPython.display import display
category_counts = jan_df["カテゴリ名"].value_counts()
print(category_counts.to_string())

カテゴリ名
トップス                6299
ボトムス                3486
ｾﾙﾌﾌﾞﾗﾝﾄﾞﾒｰｷｬｯﾌﾟ    2512
メンズトップス             2259
レディーストップス           2089
学用品                 1971
ヘアケア                1854
アウター                1722
筆記                  1653
レディースソックス           1579
単品メイク               1529
事務用品                1218
釣り具用品               1013
スキンケア               1001
女性頭髪                 992
メンズボトム               986
シーズン                 953
ヘアカラー                923
芳香消臭                 919
猫ウェット                918
メンズビューティ             913
ランジェリー＆ファンデーション      829
和洋菓子                 805
健康食品                 791
メンズソックス              747
菓子パン                 742
入浴剤                  721
芳香剤HC                715
メンズスポーツシューズ          714
エギ・ルアー               713
犬猫生活用品               673
ｶｳﾝｾﾘﾝｸﾞ化粧品          665
未登録等その他              637
珍味                   632
季節                   623
犬スナック                620
柔軟剤                  592
食器                   591
レディーススポーツシューズ        588
スナック               

In [11]:
def create_session_feature(input_session_df: pd.DataFrame) -> pd.DataFrame:
    use_columns = [
        "時刻"
    ]

    return input_session_df[use_columns].copy()

In [12]:
create_session_feature(train_session_df)

,時刻
0,0
1,0
2,0
3,0
4,0
...,...
2120866,23
2120867,23
2120868,23
2120869,23


In [13]:
!pip install holidays

In [14]:
import holidays
from datetime import datetime

def create_date_feature(input_session_df):
    jp_holidays = holidays.Japan(years=datetime.now().year)
    date = pd.to_datetime(input_session_df["売上日"])

    output_df = pd.DataFrame()

    def is_holiday(date):
      return date in jp_holidays


    #output_df["年"] = date.dt.year
    output_df["曜日"] = date.dt.dayofweek
    output_df["日"] = date.dt.day
    output_df["祝日"] = date.apply(is_holiday)
    output_df['週末'] = output_df['曜日'].apply(lambda x: 1 if x in [5, 6] else 0)

    return output_df

In [15]:
create_date_feature(train_session_df)

,曜日,日,祝日,週末
0,0,1,False,0
1,0,1,False,0
2,0,1,False,0
3,0,1,False,0
4,0,1,False,0
...,...,...,...,...
2120866,3,31,False,0
2120867,3,31,False,0
2120868,3,31,False,0
2120869,3,31,False,0


In [16]:
input_df = train_session_df.copy()

# 性別がとりうる値を最初に決める
categories = train_session_df["性別"].unique()

# 列にカテゴリを指定して, 性別の種類ごとに存在していたら 1 しないと 0 のデータに変換
pd.get_dummies(input_df["性別"], columns=categories)

,不明,女性,男性
0,True,False,False
1,True,False,False
2,False,True,False
3,False,True,False
4,False,True,False
...,...,...,...
2120866,False,False,True
2120867,False,False,True
2120868,False,False,True
2120869,False,False,True


In [17]:
def create_gender_feature(input_df):

    # 性別がとりうる値を最初に決める
    categories = train_session_df["性別"].unique()

    # 列にカテゴリを指定して, 性別の種類ごとに存在していたら 1 しないと 0 のデータに変換
    output_df = pd.get_dummies(input_df["性別"], columns=categories)

    return output_df

In [18]:
create_gender_feature(test_session_df)

,不明,女性,男性
0,False,True,False
1,False,True,False
2,False,False,True
3,False,True,False
4,False,False,True
...,...,...,...
121410,True,False,False
121411,False,True,False
121412,False,True,False
121413,False,True,False


In [19]:
def create_tenpo_feature(input_df):

    # 性別がとりうる値を最初に決める
    categories = train_session_df["店舗名"].unique()

    # 列にカテゴリを指定して, 性別の種類ごとに存在していたら 1 しないと 0 のデータに変換
    output_df = pd.get_dummies(input_df["店舗名"], columns=categories)

    return output_df

In [20]:
create_tenpo_feature(test_session_df)

,つくば,新宮店,日田店,益浦店,福岡空,門司店
0,False,True,False,False,False,False
1,False,False,False,False,False,True
2,False,True,False,False,False,False
3,False,False,True,False,False,False
4,False,False,False,False,False,True
...,...,...,...,...,...,...
121410,False,True,False,False,False,False
121411,False,False,True,False,False,False
121412,False,False,True,False,False,False
121413,False,False,True,False,False,False


In [21]:
def create_nendai_feature(input_df):

    # 性別がとりうる値を最初に決める
    categories = train_session_df["年代"].unique()

    # 列にカテゴリを指定して, 性別の種類ごとに存在していたら 1 しないと 0 のデータに変換
    output_df = pd.get_dummies(input_df["年代"], columns=categories)

    return output_df

In [22]:
create_nendai_feature(test_session_df)

,10代以下,20代,30代,40代,50代,60代,70代,80代以上,不明
0,False,False,False,True,False,False,False,False,False
1,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,True,False,False,False
4,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...
121410,False,False,False,True,False,False,False,False,False
121411,False,False,False,False,False,False,True,False,False
121412,False,False,False,False,True,False,False,False,False
121413,False,False,False,False,True,False,False,False,False


In [23]:
class CountFeature:
    def __init__(self, target_column: str):
        self.target_column = target_column

    def __call__(self, input_df):
        vc = train_session_df[self.target_column].value_counts()

        output_df = pd.DataFrame({
            self.target_column: input_df[self.target_column].map(vc)
        })

        return output_df.add_prefix("Count_")

In [24]:
def create_tenposu_feature(input_df):

    output_df = pd.DataFrame()
    # 列にカテゴリを指定して, 店舗数の種類ごとに存在していたら 1 しないと 0 のデータに変換
    output_df = CountFeature("店舗名")(test_session_df)

    return output_df

In [25]:
CountFeature("店舗名")(test_session_df)

,Count_店舗名
0,693866
1,293743
2,693866
3,189436
4,293743
...,...
121410,693866
121411,189436
121412,189436
121413,189436


In [26]:
def create_kokyakucd_feature(input_df):

    output_df = pd.DataFrame()
    # 列にカテゴリを指定して, 顧客CDの種類ごとに存在していたら 1 しないと 0 のデータに変換
    output_df = CountFeature("顧客CD")(test_session_df)

    return output_df

In [27]:
#それぞれの顧客CDが何回そこに表れたかが出力される
CountFeature("顧客CD")(test_session_df)

,Count_顧客CD
0,20.0
1,33.0
2,18.0
3,NaN
4,6.0
...,...
121410,33.0
121411,1.0
121412,15.0
121413,7.0


In [28]:
TARGET_COLUMNS = ['チョコレート', 'ビール', 'ヘアケア', '米（5㎏以下）']

In [ ]:
# janコード → カテゴリ名に変換する辞書を作って
jan2category = jan_df.set_index("JAN")["カテゴリ名"]

# ログデータの jan コードをカテゴリに直す
category_names = train_log_df["JAN"].map(jan2category).rename("カテゴリ名")

# 予測対象のカテゴリが入っているとリークになる可能性があるので除外
idx_rm = category_names.isin(TARGET_COLUMNS)
category_names = category_names[~idx_rm].reset_index(drop=True)

In [ ]:
#session_idから顧客CDの辞書を作る
session2customer = train_session_df.set_index("session_id")["顧客CD"]
#renameしてindexを付け直す．リークしないように
_customers = train_log_df["session_id"].map(session2customer).rename("顧客CD")[~idx_rm].reset_index(drop=True)

In [ ]:
customer_category_total_count_df = train_log_df[~idx_rm]\
    .groupby([_customers, category_names])["売上数量"]\
    .sum().unstack()

customer_category_total_count_df = customer_category_total_count_df.fillna(0).astype(int)

In [ ]:
from sklearn.decomposition import TruncatedSVD

In [ ]:
clf = TruncatedSVD(n_components=30)
z = clf.fit_transform(customer_category_total_count_df.values)
svd_customer_df = pd.DataFrame(z, index=customer_category_total_count_df.index)
svd_customer_df.head()

In [ ]:
def create_customer_purchase_feature(input_df):
    """この実装では svd で圧縮した購買履歴データがある前提で記述をしています"""

    output_df = pd.merge(input_df["顧客CD"], svd_customer_df, on="顧客CD", how="left").drop(columns=["顧客CD"])
    return output_df.add_prefix("customer_purchase_")

In [ ]:
train_log_df = pl.from_pandas(train_log_df)
jan_df = pl.from_pandas(jan_df)
train_session_df = pl.from_pandas(train_session_df)

In [ ]:
def create_category_matrix(
    train_log_df,
    jan_df,
):
  """
  縦軸session_id, 横軸カテゴリ名のmatrixを作る
  """
  category_df = train_log_df.join(
        jan_df, on="JAN", how="left"
  ).group_by(["session_id", "カテゴリ名"]).agg(
      pl.col("売上数量").sum().alias("売上数量")
  )

  category_matrix_df = category_df.pivot(
      values="売上数量",
      index="session_id",
      columns="カテゴリ名",
      aggregate_function="sum"
  )

  # 売上数量を0, 1に収める

  category_matrix_df = category_matrix_df.with_columns([
      pl.col(col).clip(0,1)
      for col in category_matrix_df.columns if col != "session_id"
  ]).fill_null(0)

  return category_matrix_df

purchase_ratio_df = pl.concat(
    [
        train_session_df[["session_id", "店舗名", "年代", "性別", "顧客CD"]],
        create_category_matrix(train_log_df, jan_df)[Ctarget_col_list]
    ], how="horizontal"
)

# purchase_ratio_df の作成
purchase_ratio_df = pd.concat(
    [
        train_session_df[["session_id", "店舗名", "年代", "性別", "顧客CD"]],
        create_category_matrix(train_log_df, jan_df)[target_col_list]
    ], axis=1
)


In [ ]:
def purchase_ratio_feature(input_session_df, purchase_ratio_df):
    # `session_id` で結合
    merged_df = input_session_df.merge(purchase_ratio_df, on="session_id", how="left")

    # 顧客CDごとの購入比率
    customer_purchase_ratio_df = merged_df.groupby("顧客CD")[CFG.target_col_list].mean()
    customer_purchase_ratio_df = customer_purchase_ratio_df.add_prefix("顧客CD_purchase_ratio_").reset_index()

    # 性別ごとの購入比率
    sex_purchase_ratio_df = merged_df.groupby("性別")[CFG.target_col_list].mean()
    sex_purchase_ratio_df = sex_purchase_ratio_df.add_prefix("性別_purchase_ratio_").reset_index()

    # 年代ごとの購入比率
    age_purchase_ratio_df = merged_df.groupby("年代")[CFG.target_col_list].mean()
    age_purchase_ratio_df = age_purchase_ratio_df.add_prefix("年代_purchase_ratio_").reset_index()

    # 店舗名ごとの購入比率
    shop_purchase_ratio_df = merged_df.groupby("店舗名")[CFG.target_col_list].mean()
    shop_purchase_ratio_df = shop_purchase_ratio_df.add_prefix("店舗名_purchase_ratio_").reset_index()

    return customer_purchase_ratio_df, sex_purchase_ratio_df, age_purchase_ratio_df, shop_purchase_ratio_df

In [ ]:
customer_purchase_ratio_df, sex_purchase_ratio_df, age_purchase_ratio_df, shop_purchase_ratio_df = purchase_ratio_feature(train_input_session_df, purchase_ratio_df)

In [49]:
from typing import List
from contextlib import contextmanager
from time import time

# コードの実行時間を計測する便利関数
# https://github.com/nyk510/vivid/blob/master/vivid/utils.py
class Timer:
    def __init__(self, logger=None, format_str="{:.3f}[s]", prefix=None, suffix=None, sep=" "):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

def build_feature(input_df: pd.DataFrame, feature_functions: List) -> pd.DataFrame:
    # 出力するデータフレームを空で用意して
    out_df = pd.DataFrame()

    print("start build features...")

    # 各特徴生成関数ごとで
    for func in feature_functions:

        with Timer(prefix=f"create {func.__name__}"):
            # 特徴量を作成し
            _df = func(input_df)

        # 横方向 (axis=1) にがっちゃんこ (concat) する
        out_df = pd.concat([out_df, _df], axis=1)

    return out_df

In [64]:
functions = [
    create_session_feature,
    create_gender_feature,
    create_date_feature,
    create_tenpo_feature,
    create_nendai_feature,
    create_tenposu_feature,
    create_kokyakucd_feature,
    create_customer_purchase_feature,
]

In [65]:
with Timer(prefix="build train..."):
    train_feat_df = build_feature(train_session_df, feature_functions=functions)

with Timer(prefix="build test..."):
    test_feat_df = build_feature(test_session_df, feature_functions=functions)

start build features...
create create_session_feature 0.020[s]
create create_gender_feature 0.328[s]
create create_date_feature 7.234[s]
create create_tenpo_feature 0.256[s]
create create_nendai_feature 0.237[s]
create create_tenposu_feature 0.175[s]
create create_kokyakucd_feature 1.348[s]
create create_customer_purchase_feature 3.284[s]
build train... 13.448[s]
start build features...
create create_session_feature 0.003[s]
create create_gender_feature 0.183[s]
create create_date_feature 0.372[s]
create create_tenpo_feature 0.111[s]
create create_nendai_feature 0.104[s]
create create_tenposu_feature 0.178[s]
create create_kokyakucd_feature 1.332[s]
create create_customer_purchase_feature 0.251[s]
build test... 2.560[s]


In [66]:
feat = pd.read_csv('/content/drive/MyDrive/atma/feature_train_2.csv')


In [67]:
feat_test = pd.read_csv('/content/drive/MyDrive/atma/feature_test_2.csv')

In [68]:
feat

,choco_feature,bear_feature,haircare_feature,rice_feature
0,1.286275,1.020426,1.202358,0.924809
1,1.286275,1.020426,1.202358,0.924809
2,1.286275,0.985439,0.852432,0.752939
3,1.286275,0.985439,0.852432,0.752939
4,1.286275,0.985439,0.852432,0.752939
...,...,...,...,...
2120546,1.554015,1.897273,1.614320,1.983749
2120547,1.554015,1.897273,1.614320,1.983749
2120548,1.554015,1.897273,1.614320,1.983749
2120549,1.554015,1.897273,1.614320,1.983749


In [69]:
train_feat_df = pd.concat([train_feat_df, feat], axis=1)

In [70]:
train_feat_df

,時刻,不明,女性,男性,曜日,日,祝日,つくば,新宮店,日田店,...,customer_purchase_24,customer_purchase_25,customer_purchase_26,customer_purchase_27,customer_purchase_28,customer_purchase_29,choco_feature,bear_feature,haircare_feature,rice_feature
0,0,True,False,False,0,1,False,True,False,False,...,-1.836609,0.056516,-1.003705,1.197711,0.330107,-0.177918,1.286275,1.020426,1.202358,0.924809
1,0,True,False,False,0,1,False,True,False,False,...,1.682822,-0.546055,-4.539963,-0.526346,1.292873,0.751415,1.286275,1.020426,1.202358,0.924809
2,0,False,True,False,0,1,False,True,False,False,...,1.830641,6.269339,-2.047747,6.720994,2.081028,-3.806281,1.286275,0.985439,0.852432,0.752939
3,0,False,True,False,0,1,False,True,False,False,...,1.528740,0.628689,0.445982,-1.265130,-0.436215,-2.413832,1.286275,0.985439,0.852432,0.752939
4,0,False,True,False,0,1,False,True,False,False,...,-7.773605,1.821006,-2.009421,-6.437783,-1.534269,9.068636,1.286275,0.985439,0.852432,0.752939
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2120546,23,False,False,True,3,31,False,False,False,False,...,-0.021917,0.075598,-0.007400,-0.000587,0.094081,0.016387,1.554015,1.897273,1.614320,1.983749
2120547,23,False,False,True,3,31,False,False,False,False,...,-0.250057,-0.214091,0.316939,-0.314654,-0.031169,-0.168987,1.554015,1.897273,1.614320,1.983749
2120548,23,False,False,True,3,31,False,False,False,False,...,-0.003753,0.002975,-0.013249,0.006543,0.009306,0.010949,1.554015,1.897273,1.614320,1.983749
2120549,23,False,False,True,3,31,False,False,False,False,...,-4.296553,6.642505,2.369098,-9.055719,2.809552,-21.865950,1.554015,1.897273,1.614320,1.983749


In [71]:
test_feat_df = pd.concat([test_feat_df, feat_test], axis=1)

In [72]:
assert train_feat_df.columns.equals(test_feat_df.columns)

In [73]:
test_feat_df.head()

,時刻,不明,女性,男性,曜日,日,祝日,つくば,新宮店,日田店,...,customer_purchase_24,customer_purchase_25,customer_purchase_26,customer_purchase_27,customer_purchase_28,customer_purchase_29,choco_feature,bear_feature,haircare_feature,rice_feature
0,17,False,True,False,6,3,True,False,True,False,...,0.098578,6.085979,-2.153290,-6.624559,-0.321624,-9.945392,1.600709,0.808032,0.808032,0.964582
1,17,False,True,False,4,1,False,False,False,False,...,-3.347201,0.443796,-1.701189,-0.060082,0.702007,1.515831,1.242665,1.035536,1.035536,4.740614
2,11,False,False,True,3,7,False,False,True,False,...,-4.053895,-5.444377,0.250034,-1.641328,1.423383,-2.210929,1.258484,1.988388,1.988388,4.834450
3,8,False,True,False,4,22,False,False,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,1.407995,1.628637,1.628637,3.221336
4,15,False,False,True,5,9,False,False,False,False,...,-1.382703,0.444577,0.523266,-1.120246,-0.665612,-0.993565,2.485013,1.654474,1.654474,1.315418


# 正解ラベル

In [74]:
TARGET_COLUMNS = ['チョコレート', 'ビール', 'ヘアケア', '米（5㎏以下）']

In [75]:
# def create_target_data(train_session_df, log_df):
#     output_df = pd.DataFrame()
#     for c in TARGET_COLUMNS:
#         print(f"\t- start {c}")
#         idx = jan_df["カテゴリ名"] == c
#         target_jan = jan_df["JAN"][idx].values
#         is_target = log_df["JAN"].isin(target_jan)

#         # セッションごとに, 対象商品のレコードの売上個数を合計して
#         total_amount_by_session = log_df[is_target].groupby("session_id")["売上数量"].sum()

#         # 1 個以上購入があったセッションを positive (購入あり) セッションとみなし
#         positive_sessions = total_amount_by_session[total_amount_by_session > 0].index

#         # 学習用データのセッションと比較・含まれているとき 1 そうでないとき 0 になるようにする
#         y_true = train_session_df["session_id"].isin(positive_sessions).astype(int)

#         output_df[c] = y_true

#     return output_df

In [76]:
def create_target_data(train_session_df, log_df, jan_df, threshold=1, price_threshold=None):
    output_df = pd.DataFrame()

    # 負の売上数量を削除（返品を除外）
    log_df = log_df[log_df["売上数量"] >= 0]

    for c in TARGET_COLUMNS:
        print(f"\t- start {c}")
        idx = jan_df["カテゴリ名"] == c
        target_jan = jan_df["JAN"][idx].values
        is_target = log_df["JAN"].isin(target_jan)

        # セッションごとの売上個数合計
        total_amount_by_session = log_df[is_target].groupby("session_id")["売上数量"].sum()

        # 売上金額のしきい値がある場合は、金額ベースのフィルタ
        if price_threshold is not None:
            total_sales_by_session = log_df[is_target].groupby("session_id")["売上金額"].sum()
            positive_sessions = total_sales_by_session[total_sales_by_session >= price_threshold].index
        else:
            positive_sessions = total_amount_by_session[total_amount_by_session >= threshold].index

        # 学習用データのセッションと比較
        y_true = train_session_df["session_id"].isin(positive_sessions).astype(int)

        output_df[c] = y_true

    return output_df

In [77]:
with Timer(prefix="create target..."):
    target_df = create_target_data(train_session_df=train_session_df, log_df=train_log_df,jan_df=jan_df)

	- start チョコレート
	- start ビール
	- start ヘアケア
	- start 米（5㎏以下）
create target... 7.919[s]


In [78]:
target_df.mean()

,0
チョコレート,0.068832
ビール,0.074139
ヘアケア,0.028059
米（5㎏以下）,0.019925


In [105]:
train_target_df.mean()

,0
チョコレート,0.068940
ビール,0.074257
ヘアケア,0.028136
米（5㎏以下）,0.019964


In [79]:
from sklearn.model_selection import KFold

fold = KFold(n_splits=5, shuffle=True, random_state=510)
cv = list(fold.split(train_feat_df.values, train_target_df.iloc[:, 0].values))

In [80]:
from sklearn.metrics import roc_auc_score
import lightgbm as lgbm

def fit_lgbm(X,
             y,
             cv,
             params: dict=None,
             verbose: int=50):
    """lightGBM を CrossValidation の枠組みで学習を行なう function"""

    # パラメータがないときは、空の dict で置き換える
    if params is None:
        params = {}

    models = []
    n_records = len(X)
    # training data の target と同じだけのゼロ配列を用意
    oof_pred = np.zeros((n_records, ), dtype=np.float32)

    for i, (idx_train, idx_valid) in enumerate(cv):
        # この部分が交差検証のところです。データセットを cv instance によって分割します
        # training data を trian/valid に分割
        x_train, y_train = X[idx_train], y[idx_train]
        x_valid, y_valid = X[idx_valid], y[idx_valid]

        clf = lgbm.LGBMClassifier(**params)

        with Timer(prefix="fit fold={} ".format(i)):
            clf.fit(x_train, y_train,
                    eval_set=[(x_valid, y_valid)],
                    callbacks=[lgbm.callback.early_stopping(stopping_rounds=100),
                      lgbm.log_evaluation(period=verbose),],)

        # 予測確率を出力してもらう
        pred_i = clf.predict_proba(x_valid)
        # output の形は shape = (n_samples, 2) になっていることに注意!
        y_pred_prob = pred_i[:, 1]

        oof_pred[idx_valid] = y_pred_prob
        models.append(clf)

        # 今回の指標の MAE で計算する
        score = roc_auc_score(y_valid, y_pred_prob, )
        print(f" - fold{i} - {score:.4f}")

    return oof_pred, models

In [81]:
params = {
    # 目的関数. これの意味で最小となるようなパラメータを探します.
    # 今回は 0-1 問題なので, よく使われる binary を利用します
    "objective": "binary",

     # 学習率. 小さいほどなめらかな決定境界が作られて性能向上に繋がる場合が多いです、
    # がそれだけ木を作るため学習に時間がかかります
    "learning_rate": .1,

    # 木の最大数. early_stopping という枠組みで木の数は制御されるようにしていますのでとても大きい値を指定しておきます.
    "n_estimators": 10000,

    # 特徴重要度計算のロジック(後述)
    "importance_type": "gain",
    "random_state": 510,

    # ログがあまりに多いので少なめに.
    "verbose": -1
}
#futre work hypper parameter

In [ ]:
# 今回は予測対象が複数あるので for 文を使ってそれぞれの列に対して予測をしてくれるモデルを作ります.
results = []

for col_i in TARGET_COLUMNS:
    with Timer(prefix=f"column={col_i}"):
        oof, models = fit_lgbm(X=train_feat_df.values, y=target_df[col_i].values, cv=cv, params=params)

    results.append([oof, models])

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[50]	valid_0's binary_logloss: 0.231267
[100]	valid_0's binary_logloss: 0.228179
[150]	valid_0's binary_logloss: 0.226131
[200]	valid_0's binary_logloss: 0.224671
[250]	valid_0's binary_logloss: 0.223545
[300]	valid_0's binary_logloss: 0.22257
[350]	valid_0's binary_logloss: 0.221797
[400]	valid_0's binary_logloss: 0.221202
[450]	valid_0's binary_logloss: 0.220606
[500]	valid_0's binary_logloss: 0.22011
[550]	valid_0's binary_logloss: 0.219687
[600]	valid_0's binary_logloss: 0.21932
[650]	valid_0's binary_logloss: 0.218928
[700]	valid_0's binary_logloss: 0.218596
[750]	valid_0's binary_logloss: 0.218213
[800]	valid_0's binary_logloss: 0.217906
[850]	valid_0's binary_logloss: 0.217619
[900]	valid_0's binary_logloss: 0.217374
[950]	valid_0's binary_logloss: 0.217115
[1000]	valid_0's binary_logloss: 0.216968
[1050]	valid_0's binary_logloss: 0.216773
[1100]	valid_0's binary_logloss: 0.216609
[1150]	valid_0's binary_logloss: 0.21

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 - fold0 - 0.7736


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

Training until validation scores don't improve for 100 rounds
[50]	valid_0's binary_logloss: 0.0916268
[100]	valid_0's binary_logloss: 0.0907466
[150]	valid_0's binary_logloss: 0.0902995
[200]	valid_0's binary_logloss: 0.0900386
[250]	valid_0's binary_logloss: 0.0898508
[300]	valid_0's binary_logloss: 0.0897155
[350]	valid_0's binary_logloss: 0.0895788
[400]	valid_0's binary_logloss: 0.0895117
[450]	valid_0's binary_logloss: 0.0894214
[500]	valid_0's binary_logloss: 0.0893641
[550]	valid_0's binary_logloss: 0.0892894
[600]	valid_0's binary_logloss: 0.0891963
[650]	valid_0's binary_logloss: 0.0891596
[700]	valid_0's binary_logloss: 0.089097
[750]	valid_0's binary_logloss: 0.0889993
[800]	valid_0's binary_logloss: 0.0889847
[850]	valid_0's binary_logloss: 0.0889245
[900]	valid_0's binary_logloss: 0.0889241
[950]	valid_0's binary_logloss: 0.0888827
[1000]	valid_0's binary_logloss: 0.0888443
[1050]	valid_0's binary_logloss: 0.0888132
[1100]	valid_0's binary_logloss: 0.0888147
[1150]	valid_

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


 - fold3 - 0.7366


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[50]	valid_0's binary_logloss: 0.0926765
[100]	valid_0's binary_logloss: 0.0918234
[150]	valid_0's binary_logloss: 0.0914118
[200]	valid_0's binary_logloss: 0.0911633


In [ ]:
oof_df = pd.DataFrame()

for col_i, (oof, _) in zip(TARGET_COLUMNS, results):
    oof_df[col_i] = oof

In [ ]:
roc_auc_score(target_df.values, oof_df.values)

In [ ]:
train_feat_df.head()

In [ ]:
def visualize_importance(models, feat_train_df):
    """lightGBM の model 配列の feature importance を plot する
    CVごとのブレを boxen plot として表現します.

    args:
        models:
            List of lightGBM models
        feat_train_df:
            学習時に使った DataFrame
    """
    feature_importance_df = pd.DataFrame()
    for i, model in enumerate(models):
        _df = pd.DataFrame()
        _df["feature_importance"] = model.feature_importances_
        _df["column"] = feat_train_df.columns
        _df["fold"] = i + 1
        feature_importance_df = pd.concat([feature_importance_df, _df],
                                          axis=0, ignore_index=True)

    order = feature_importance_df.groupby("column")\
        .sum()[["feature_importance"]]\
        .sort_values("feature_importance", ascending=False).index[:50]

    fig, ax = plt.subplots(figsize=(8, max(6, len(order) * .25)))
    sns.boxenplot(data=feature_importance_df,
                  x="feature_importance",
                  y="column",
                  order=order,
                  ax=ax,
                  palette="viridis",
                  orient="h")
    ax.tick_params(axis="x", rotation=90)
    ax.set_title("Importance")
    ax.grid()
    fig.tight_layout()
    return fig, ax

In [ ]:
for col_i, (oof, models) in zip(TARGET_COLUMNS, results):
    fig, ax = visualize_importance(models, train_feat_df)
    ax.set_title(col_i)

In [ ]:
submission_df = pd.DataFrame()

for col_i, (oof, models) in zip(TARGET_COLUMNS, results):
    predicts = [model.predict_proba(test_feat_df.values)[:, 1] for model in models]
    predict_i = np.array(predicts).mean(axis=0)

    submission_df[col_i] = predict_i

In [ ]:
for c in TARGET_COLUMNS:
    fig, ax = plt.subplots(figsize=(8, 5))

    sns.histplot(oof_df[c], ax=ax, label="OOF", kde=True)
    sns.histplot(submission_df[c], ax=ax, label="Test", kde=True)

    ax.set_xlabel("予測値", fontproperties=font_prop)  # x軸ラベルの文字化けを防ぐ
    ax.set_ylabel("Count", fontproperties=font_prop)  # y軸ラベルも設定
    ax.set_title(c, fontproperties=font_prop)  # 日本語タイトル対応
    ax.legend()
    ax.grid()

    plt.show()

In [ ]:
import os

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/atma/tutorial_1"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
submission_df.to_csv(os.path.join(OUTPUT_DIR, "#7__submission.csv"), index=False)

In [51]:
rm -rf ~/.cache/matplotlib